In [4]:
###########################################
# BASE IMAGES — fully windowed version
# Never loads full raster into memory.
# Outputs are sorted into per track-frame folders.
###########################################

import re
import numpy as np
import rasterio
from rasterio.windows import Window
from pathlib import Path
from PIL import Image

folder = Path("/home/matt/NISAR_cal_eval/Wolf")   # <------ change this
out_dir = folder / "base_images"
out_dir.mkdir(exist_ok=True)

tif_paths = sorted(folder.glob("*.tif"))

# FILENAME PATTERN
# e.g. NISAR_L2_PR_GSLC_010_026_D_088_2005_SHSH_A_20251124T002849_20251124T002910_P05006_N_P_J_001_HH_freqA.tif
#                          ^^^ track   ^^^ frame
track_frame_pattern = re.compile(r".*_([0-9]{3})_(?:A|D)_([0-9]{3})_")


def get_track_frame(path):
    """
    Pull track and frame out of the filename. Returns (track, frame),
    or (None, None) if the filename doesn't match the expected pattern.
    """
    m = track_frame_pattern.match(path.name)
    if m:
        return m.group(1), m.group(2)
    return None, None


def compute_percentiles_streaming(src, band=1, step=1024):
    """
    Compute percentiles by sampling windows across the raster.
    Avoids loading the full image.
    """
    samples = []

    height, width = src.height, src.width

    for row in range(0, height, step):
        for col in range(0, width, step):
            win = Window(col, row,
                          min(step, width - col),
                          min(step, height - row))

            block = src.read(band, window=win)
            mag = np.abs(block)
            db = 10 * np.log10(mag + 1e-6)
            valid = db[mag > 0]

            if valid.size > 0:
                samples.append(valid)

    if not samples:
        return 0, 1

    samples = np.concatenate(samples)
    return np.nanpercentile(samples, 2), np.nanpercentile(samples, 98)


for path in tif_paths:

    track, frame = get_track_frame(path)
    if track is None:
        print(f"Skipping unmatched filename: {path.name}")
        continue

    # One output folder per track-frame pair
    tf_dir = out_dir / f"{track}_{frame}"
    tf_dir.mkdir(exist_ok=True)

    with rasterio.open(path) as src:
        profile = src.profile
        height, width = src.height, src.width

        # Compute vmin/vmax using streaming sampling
        vmin, vmax = compute_percentiles_streaming(src)

        # ---- Write TIF (windowed) ----
        out_tif = tf_dir / f"{path.stem}.tif"
        profile.update(dtype=rasterio.float32, count=1)

        with rasterio.open(out_tif, "w", **profile) as dst:
            step = 1024
            for row in range(0, height, step):
                for col in range(0, width, step):
                    win = Window(col, row,
                                 min(step, width - col),
                                 min(step, height - row))

                    block = src.read(1, window=win)
                    mag = np.abs(block)
                    db = 10 * np.log10(mag + 1e-6)

                    dst.write(db.astype(np.float32), 1, window=win)

        # ---- Write PNG (windowed) ----
        # Create an empty grayscale image
        out_png = tf_dir / f"{path.stem}.png"
        png_img = Image.new("L", (width, height))

        step = 1024
        for row in range(0, height, step):
            for col in range(0, width, step):
                win = Window(col, row,
                             min(step, width - col),
                             min(step, height - row))

                block = src.read(1, window=win)
                mag = np.abs(block)
                db = 10 * np.log10(mag + 1e-6)

                # Normalize to 0-255
                scaled = np.clip((db - vmin) / (vmax - vmin), 0, 1)
                scaled = (scaled * 255).astype(np.uint8)

                # Convert to PIL image and paste into final PNG
                tile = Image.fromarray(scaled)
                png_img.paste(tile, (col, row))

        png_img.save(out_png)

/tmp/ipykernel_243563/2368457628.py:52: DeprecationWarning: Setting the shape on a NumPy array has been deprecated in NumPy 2.5.
As an alternative, you can create a new view using np.reshape (with copy=False if needed).
  block = src.read(band, window=win)
/tmp/ipykernel_243563/2368457628.py:97: DeprecationWarning: Setting the shape on a NumPy array has been deprecated in NumPy 2.5.
As an alternative, you can create a new view using np.reshape (with copy=False if needed).
  block = src.read(1, window=win)
/tmp/ipykernel_243563/2368457628.py:115: DeprecationWarning: Setting the shape on a NumPy array has been deprecated in NumPy 2.5.
As an alternative, you can create a new view using np.reshape (with copy=False if needed).
  block = src.read(1, window=win)


In [ ]:
###########################################
# 
# Nearest Neighbors RATIOS- converts downloaded tifs into nearest neighbors amplitude ratios, pngs and tifs
#
########################################## 
import numpy as np
from pathlib import Path
import re
import rasterio
from rasterio.warp import reproject, Resampling
from datetime import datetime

# CONFIG
gslcPath = Path("path/to/where/GSLCs/were/downloaded")   #<------ NOTE: change this
workDir  = gslcPath /'ratios/'
workDir.mkdir(exist_ok=True)

# FILENAME PATTERN
pattern = re.compile(
    r".*_(A|D)_[^_]*_[^_]*_[^_]*_[^_]*_"
    r"([0-9]{8}T[0-9]{6})_"
    r"([0-9]{8}T[0-9]{6})_"
    r"([A-Za-z][0-9]{5})_"
    r".*_(HH|HV|VH|VV)"
    r"(?:_freq([AB]))?"
    r"\.tif$"
)

# DISCOVER AND GROUP TIFS
entries = []
for tif in sorted(gslcPath.glob("*.tif")):
    m = pattern.match(tif.name)
    if m:
        look      = m.group(1)
        timestamp = m.group(2)
        pol       = m.group(5)
        freq      = m.group(6)  # 'A', 'B', or None if absent
        date      = datetime.strptime(timestamp[:8], '%Y%m%d')
        entries.append((tif, look, date, pol, freq))

if not entries:
    raise RuntimeError(f"No matching .tif files found in {gslcPath}")

groups = {}
for tif, look, date, pol, freq in entries:
    groups.setdefault(look, {})
    groups[look].setdefault(pol, [])
    groups[look][pol].append((tif, date))

# PROCESS EACH LOOK / POL GROUP
for look_dir, pol_groups in groups.items():
    for pol, items in pol_groups.items():

        items = sorted(items, key=lambda x: x[1])
        nslc  = len(items)

        if nslc < 2:
            print(f"Skipping {look_dir} {pol}: only one image")
            continue

        # Use first file as reference grid
        with rasterio.open(items[0][0]) as ref:
            ny, nx        = ref.height, ref.width
            ref_transform = ref.transform
            ref_crs       = ref.crs

        # Load amplitude stack, aligned to reference grid
        amps = np.zeros((nslc, ny, nx), dtype='float32')

        for i, (tif, date) in enumerate(items):
            with rasterio.open(tif) as src:
                if i == 0:
                    amps[i] = np.abs(src.read(1))
                else:
                    aligned = np.zeros((ny, nx), dtype='float32')
                    reproject(
                        source        = np.abs(src.read(1)),
                        destination   = aligned,
                        src_transform = src.transform,
                        src_crs       = src.crs,
                        dst_transform = ref_transform,
                        dst_crs       = ref_crs,
                        resampling    = Resampling.bilinear
                    )
                    amps[i] = aligned

        # Compute nearest-neighbor amplitude ratios
        with np.errstate(divide='ignore', invalid='ignore'):
            ratios = np.where(amps[1:] > 0, amps[0:-1] / amps[1:], np.nan)

        ni = ratios.shape[0]

        # Plot and save each pair
        for i in range(ni):
            date1 = items[i][1]
            date2 = items[i+1][1]
            t1    = date1.strftime('%Y%m%d')
            t2    = date2.strftime('%Y%m%d')

            if t1 == t2:
                print(f"Skipping same-date pair: {t1} → {t2}")
                continue

            ratio = ratios[i]
            valid = ratio[np.isfinite(ratio) & (ratio > 0)]

            vmin = np.nanpercentile(valid, 2)
            vmax = np.nanpercentile(valid, 98)

            out_tif = workDir / f"ratio_{look_dir}_{pol}_{t1}_to_{t2}.tif"
            out_png = workDir / f"ratio_{look_dir}_{pol}_{t1}_to_{t2}.png"

            with rasterio.open(
                out_tif, "w",
                driver    = "GTiff",
                height    = ratio.shape[0],
                width     = ratio.shape[1],
                count     = 1,
                dtype     = ratio.dtype,
                crs       = ref_crs,
                transform = ref_transform,
            ) as dst:
                dst.write(ratio, 1)

            plt.imsave(out_png, ratio, cmap='gray_r', vmin=vmin, vmax=vmax)

In [3]:
###########################################
#
# NEAREST NEIGHBORS INTERFEROGRAMS- converts downloaded tifs into nearest neighbors interferograms, pngs and tifs
# Only pairs date-adjacent images within the same track-frame (and look direction / polarization)
#
##########################################
import numpy as np
from pathlib import Path
import re
import rasterio
from rasterio.warp import reproject, Resampling
from datetime import datetime
import matplotlib.pyplot as plt

# CONFIG
gslcPath = Path("/home/matt/NISAR_cal_eval/Wolf")   #<------ NOTE: change this
workDir  = gslcPath / 'igrams/'
workDir.mkdir(exist_ok=True)

# FILENAME PATTERN
# e.g. NISAR_L2_PR_GSLC_010_026_D_088_2005_SHSH_A_20251124T002849_20251124T002910_P05006_N_P_J_001_HH_freqA.tif
#                          ^^^ track   ^^^ frame
pattern = re.compile(
    r".*_([0-9]{3})_"        # track
    r"(A|D)_"                # look direction
    r"([0-9]{3})_"           # frame
    r"[^_]*_[^_]*_[^_]*_"    # skip 3 fields (e.g. 2005_SHSH_A)
    r"([0-9]{8}T[0-9]{6})_"  # start timestamp
    r"([0-9]{8}T[0-9]{6})_"  # end timestamp
    r"([A-Za-z][0-9]{5})_"   # product/burst ID
    r".*_(HH|HV|VH|VV)"
    r"(?:_freq([AB]))?"      # optional frequency band suffix
    r"\.tif$"
)

# DISCOVER AND GROUP TIFS
entries = []
for tif in sorted(gslcPath.glob("*.tif")):
    m = pattern.match(tif.name)
    if m:
        track     = m.group(1)
        look      = m.group(2)
        frame     = m.group(3)
        timestamp = m.group(4)   # start time for date parsing
        pol       = m.group(7)
        freq      = m.group(8)   # 'A', 'B', or None if absent
        date      = datetime.strptime(timestamp[:8], '%Y%m%d')
        entries.append((tif, track, frame, look, date, pol, freq))
    else:
        print(f"Skipping unmatched filename: {tif.name}")

if not entries:
    raise RuntimeError(f"No matching .tif files found in {gslcPath}")

# Group by track-frame → look direction → polarization
groups = {}
for tif, track, frame, look, date, pol, freq in entries:
    key = (track, frame)
    groups.setdefault(key, {})
    groups[key].setdefault(look, {})
    groups[key][look].setdefault(pol, [])
    groups[key][look][pol].append((tif, date))

# PROCESS EACH TRACK-FRAME / LOOK / POL GROUP
for (track, frame), look_groups in groups.items():

    # One output folder per track-frame pair
    tfDir = workDir / f"{track}_{frame}"
    tfDir.mkdir(exist_ok=True)

    for look_dir, pol_groups in look_groups.items():
        for pol, items in pol_groups.items():

            items = sorted(items, key=lambda x: x[1])
            nslc  = len(items)

            if nslc < 2:
                print(f"Skipping track {track} frame {frame} {look_dir} {pol}: only one image")
                continue

            # Use first file as reference grid
            with rasterio.open(items[0][0]) as ref:
                ny, nx        = ref.height, ref.width
                ref_transform = ref.transform
                ref_crs       = ref.crs

            # Load SLC stack, aligned to reference grid
            slcs = np.zeros((nslc, ny, nx), dtype='complex64')

            for i, (tif, date) in enumerate(items):
                with rasterio.open(tif) as src:
                    if i == 0:
                        slcs[i] = src.read(1)
                    else:
                        aligned = np.zeros((ny, nx), dtype='complex64')
                        reproject(
                            source        = src.read(1),
                            destination   = aligned,
                            src_transform = src.transform,
                            src_crs       = src.crs,
                            dst_transform = ref_transform,
                            dst_crs       = ref_crs,
                            resampling    = Resampling.bilinear
                        )
                        slcs[i] = aligned

            # Compute interferogram stack (nearest-neighbor pairs only)
            igram      = slcs[0:-1, :, :] * np.conj(slcs[1:, :, :])
            ni, ny, nx = np.shape(igram)

            # Save each pair as PNG + TIF, into the track-frame folder
            for i in range(ni):
                date1 = items[i][1]
                date2 = items[i+1][1]
                t1    = date1.strftime('%Y%m%d')
                t2    = date2.strftime('%Y%m%d')

                if t1 == t2:
                    print(f"Skipping same-date pair: {t1} to {t2} (track {track} frame {frame})")
                    continue

                wrapped = np.angle(igram[i, :, :])

                out_png = tfDir / f"wrapped_{look_dir}_{pol}_{t1}_to_{t2}.png"
                out_tif = tfDir / f"wrapped_{look_dir}_{pol}_{t1}_to_{t2}.tif"

                plt.imsave(out_png, wrapped, cmap='rainbow', vmin=-np.pi, vmax=np.pi)

                with rasterio.open(
                    out_tif, "w",
                    driver    = "GTiff",
                    height    = wrapped.shape[0],
                    width     = wrapped.shape[1],
                    count     = 1,
                    dtype     = 'float32',
                    crs       = ref_crs,
                    transform = ref_transform,
                ) as dst:
                    dst.write(wrapped.astype('float32'), 1)

/tmp/ipykernel_243563/2463453415.py:93: DeprecationWarning: Setting the shape on a NumPy array has been deprecated in NumPy 2.5.
As an alternative, you can create a new view using np.reshape (with copy=False if needed).
  slcs[i] = src.read(1)
/tmp/ipykernel_243563/2463453415.py:97: DeprecationWarning: Setting the shape on a NumPy array has been deprecated in NumPy 2.5.
As an alternative, you can create a new view using np.reshape (with copy=False if needed).
  source        = src.read(1),
/tmp/ipykernel_243563/2463453415.py:97: DeprecationWarning: Setting the shape on a NumPy array has been deprecated in NumPy 2.5.
As an alternative, you can create a new view using np.reshape (with copy=False if needed).
  source        = src.read(1),
/tmp/ipykernel_243563/2463453415.py:97: DeprecationWarning: Setting the shape on a NumPy array has been deprecated in NumPy 2.5.
As an alternative, you can create a new view using np.reshape (with copy=False if needed).
  source        = src.read(1),
/tm

Skipping same-date pair: 20260312 to 20260312 (track 026 frame 088)


/tmp/ipykernel_243563/2463453415.py:93: DeprecationWarning: Setting the shape on a NumPy array has been deprecated in NumPy 2.5.
As an alternative, you can create a new view using np.reshape (with copy=False if needed).
  slcs[i] = src.read(1)
/tmp/ipykernel_243563/2463453415.py:97: DeprecationWarning: Setting the shape on a NumPy array has been deprecated in NumPy 2.5.
As an alternative, you can create a new view using np.reshape (with copy=False if needed).
  source        = src.read(1),
/tmp/ipykernel_243563/2463453415.py:97: DeprecationWarning: Setting the shape on a NumPy array has been deprecated in NumPy 2.5.
As an alternative, you can create a new view using np.reshape (with copy=False if needed).
  source        = src.read(1),
/tmp/ipykernel_243563/2463453415.py:97: DeprecationWarning: Setting the shape on a NumPy array has been deprecated in NumPy 2.5.
As an alternative, you can create a new view using np.reshape (with copy=False if needed).
  source        = src.read(1),
/tm

Skipping same-date pair: 20260321 to 20260321 (track 163 frame 002)


In [ ]:
###########################################
#
# AMPLITUDE RATIO FROM TWO SLCS - converts two SLC GeoTIFFs into an
# amplitude ratio image, saved as both PNG and TIF
#
###########################################
import numpy as np
from pathlib import Path
import rasterio
from rasterio.warp import reproject, Resampling
import matplotlib.pyplot as plt

# CONFIG — edit these paths
path1 = Path("path/to/first_slc.tif")    #<------ NOTE: change this
path2 = Path("path/to/second_slc.tif")   #<------ NOTE: change this
out_dir = Path("path/to/output/ratios")  #<------ NOTE: change this

out_dir.mkdir(parents=True, exist_ok=True)

# Use path1 as the reference grid
with rasterio.open(path1) as ref:
    ny, nx        = ref.height, ref.width
    ref_transform = ref.transform
    ref_crs       = ref.crs
    amp1          = np.abs(ref.read(1))

# Align path2 onto path1's grid
with rasterio.open(path2) as src:
    amp2 = np.zeros((ny, nx), dtype='float32')
    reproject(
        source        = np.abs(src.read(1)),
        destination   = amp2,
        src_transform = src.transform,
        src_crs       = src.crs,
        dst_transform = ref_transform,
        dst_crs       = ref_crs,
        resampling    = Resampling.bilinear
    )

# Compute amplitude ratio (path1 / path2)
with np.errstate(divide='ignore', invalid='ignore'):
    ratio = np.where(amp2 > 0, amp1 / amp2, np.nan)

valid = ratio[np.isfinite(ratio) & (ratio > 0)]
vmin  = np.nanpercentile(valid, 2)
vmax  = np.nanpercentile(valid, 98)

label = f"{path1.stem}_to_{path2.stem}"

out_tif = out_dir / f"ratio_{label}.tif"
out_png = out_dir / f"ratio_{label}.png"

with rasterio.open(
    out_tif, "w",
    driver    = "GTiff",
    height    = ratio.shape[0],
    width     = ratio.shape[1],
    count     = 1,
    dtype     = ratio.dtype,
    crs       = ref_crs,
    transform = ref_transform,
) as dst:
    dst.write(ratio, 1)

plt.imsave(out_png, ratio, cmap='gray_r', vmin=vmin, vmax=vmax)

print(f"Saved {out_tif}")
print(f"Saved {out_png}")

/tmp/ipykernel_53415/79493709.py:26: DeprecationWarning: Setting the shape on a NumPy array has been deprecated in NumPy 2.5.
As an alternative, you can create a new view using np.reshape (with copy=False if needed).
  amp1          = np.abs(ref.read(1))
/tmp/ipykernel_53415/79493709.py:32: DeprecationWarning: Setting the shape on a NumPy array has been deprecated in NumPy 2.5.
As an alternative, you can create a new view using np.reshape (with copy=False if needed).
  source        = np.abs(src.read(1)),


Saved /home/matt/NISAR_cal_eval/Kilauea/specific_ratios/ratio_NISAR_L2_PR_GSLC_009_151_A_011_4005_DHDH_A_20260107T155044_20260107T155100_X05010_N_P_J_001_HH_freqA_to_NISAR_L2_PR_GSLC_015_151_A_011_4005_DHDH_A_20260320T155046_20260320T155103_P05012_N_P_J_001_HH_freqA.tif
Saved /home/matt/NISAR_cal_eval/Kilauea/specific_ratios/ratio_NISAR_L2_PR_GSLC_009_151_A_011_4005_DHDH_A_20260107T155044_20260107T155100_X05010_N_P_J_001_HH_freqA_to_NISAR_L2_PR_GSLC_015_151_A_011_4005_DHDH_A_20260320T155046_20260320T155103_P05012_N_P_J_001_HH_freqA.png


In [4]:
###########################################
#
# INTERFEROGRAM FROM TWO GSLCS - converts two GSLCs into a wrapped  interferogram, saved as both PNG and TIF
#
###########################################
import numpy as np
from pathlib import Path
import rasterio
from rasterio.warp import reproject, Resampling
import matplotlib.pyplot as plt

# CONFIG — edit these paths
path1 = Path("/home/matt/NISAR_cal_eval/Kilauea/NISAR_L2_PR_GSLC_009_151_A_011_4005_DHDH_A_20260107T155044_20260107T155100_X05010_N_P_J_001_HH_freqA.tif")    #<------ NOTE: change this
path2 = Path("/home/matt/NISAR_cal_eval/Kilauea/NISAR_L2_PR_GSLC_015_151_A_011_4005_DHDH_A_20260320T155046_20260320T155103_P05012_N_P_J_001_HH_freqA.tif")   #<------ NOTE: change this
out_dir = Path("/home/matt/NISAR_cal_eval/Kilauea/specific_igrams")  #<------ NOTE: change this, it will make the directory if it doesnt already exist

out_dir.mkdir(parents=True, exist_ok=True)

# Use path1 as the reference grid
with rasterio.open(path1) as ref:
    ny, nx        = ref.height, ref.width
    ref_transform = ref.transform
    ref_crs       = ref.crs
    slc1          = ref.read(1)

# Align path2 onto path1's grid
with rasterio.open(path2) as src:
    slc2 = np.zeros((ny, nx), dtype='complex64')
    reproject(
        source        = src.read(1),
        destination   = slc2,
        src_transform = src.transform,
        src_crs       = src.crs,
        dst_transform = ref_transform,
        dst_crs       = ref_crs,
        resampling    = Resampling.bilinear
    )

# Compute interferogram
igram   = slc1 * np.conj(slc2)
wrapped = np.angle(igram)

label = f"{path1.stem}_to_{path2.stem}"

out_png = out_dir / f"wrapped_{label}.png"
out_tif = out_dir / f"wrapped_{label}.tif"

plt.imsave(out_png, wrapped, cmap='rainbow', vmin=-np.pi, vmax=np.pi)

with rasterio.open(
    out_tif, "w",
    driver    = "GTiff",
    height    = wrapped.shape[0],
    width     = wrapped.shape[1],
    count     = 1,
    dtype     = 'float32',
    crs       = ref_crs,
    transform = ref_transform,
) as dst:
    dst.write(wrapped.astype('float32'), 1)

print(f"Saved {out_png}")
print(f"Saved {out_tif}")

/tmp/ipykernel_53415/2321607945.py:24: DeprecationWarning: Setting the shape on a NumPy array has been deprecated in NumPy 2.5.
As an alternative, you can create a new view using np.reshape (with copy=False if needed).
  slc1          = ref.read(1)
/tmp/ipykernel_53415/2321607945.py:30: DeprecationWarning: Setting the shape on a NumPy array has been deprecated in NumPy 2.5.
As an alternative, you can create a new view using np.reshape (with copy=False if needed).
  source        = src.read(1),


Saved /home/matt/NISAR_cal_eval/Kilauea/specific_igrams/wrapped_NISAR_L2_PR_GSLC_009_151_A_011_4005_DHDH_A_20260107T155044_20260107T155100_X05010_N_P_J_001_HH_freqA_to_NISAR_L2_PR_GSLC_015_151_A_011_4005_DHDH_A_20260320T155046_20260320T155103_P05012_N_P_J_001_HH_freqA.png
Saved /home/matt/NISAR_cal_eval/Kilauea/specific_igrams/wrapped_NISAR_L2_PR_GSLC_009_151_A_011_4005_DHDH_A_20260107T155044_20260107T155100_X05010_N_P_J_001_HH_freqA_to_NISAR_L2_PR_GSLC_015_151_A_011_4005_DHDH_A_20260320T155046_20260320T155103_P05012_N_P_J_001_HH_freqA.tif


In [ ]:
# ─────────────────────────────────────────────
# GIF Generation ---- still under development, the files load very slowly
# ─────────────────────────────────────────────
import re
import numpy as np
import rasterio
from rasterio.warp import reproject, Resampling
from pathlib import Path
import imageio.v2 as imageio
from PIL import Image, ImageDraw, ImageFont
import matplotlib.pyplot as plt
import matplotlib.colors as mcolors


input_dir = Path("path/to/where/GSLCs/were/downloaded")   #<------ NOTE: change this
out_dir   = input_dir / 'gif/'
out_dir.mkdir(exist_ok=True)

frame_duration = 0.5
cmap_name = "YlGnBu"
vmin, vmax = None, None
pct_low, pct_high = 2, 98
sample_size = 200_000   # max pixels sampled per file when computing percentiles

pattern = re.compile(
    r".*_(A|D)_[^_]*_[^_]*_[^_]*_[^_]*_"
    r"([0-9]{8}T[0-9]{6})_"
    r"([0-9]{8}T[0-9]{6})_"
    r"([A-Za-z][0-9]{5})_"
    r".*_(HH|HV|VH|VV)"
    r"(?:_freq([AB]))?"
    r"\.tif$"
)

def format_label(date_str):
    return f"{date_str[0:4]}-{date_str[4:6]}-{date_str[6:8]}"

def read_band_as_real(path):
    with rasterio.open(path) as src:
        arr = src.read(1)
        transform, crs = src.transform, src.crs
    if np.iscomplexobj(arr):
        arr = np.abs(arr)
    return arr.astype('float32'), transform, crs

# ─────────────────────────────────────────────
# DISCOVER + GROUP
# ─────────────────────────────────────────────
groups = {}
for f in sorted(input_dir.glob("*.tif")):
    m = pattern.match(f.name)
    if not m:
        continue
    look = m.group(1)
    pol  = m.group(5)   # adjusted index per your fix
    date = m.group(2)[:8]
    groups.setdefault((look, pol), []).append((f, date))

if not groups:
    raise RuntimeError(f"No files matched the expected filename pattern in {input_dir}")

print(f"Found {len(groups)} (look, pol) group(s): {list(groups.keys())}")

# ─────────────────────────────────────────────
# PROCESS EACH GROUP INDEPENDENTLY
# ─────────────────────────────────────────────
for (look, pol), items in groups.items():
    items = sorted(items, key=lambda x: x[1])
    n = len(items)
    if n == 0:
        continue

    print(f"\n=== Group: look={look}, pol={pol} — {n} files ===")

    # ---- PASS 1: reference grid + percentile range (streaming, no full stack) ----
    ref_transform, ref_crs, ref_shape = None, None, None
    samples = []

    for f, date in items:
        arr, transform, crs = read_band_as_real(f)

        if ref_transform is None:
            ref_transform, ref_crs, ref_shape = transform, crs, arr.shape

        valid = arr[np.isfinite(arr) & (arr >= 0)]
        if len(valid) > sample_size:
            valid = np.random.choice(valid, sample_size, replace=False)
        samples.append(valid)

        del arr  # free immediately

    all_samples = np.concatenate(samples) if samples else np.array([])
    del samples

    group_vmin, group_vmax = vmin, vmax
    if (group_vmin is None or group_vmax is None) and len(all_samples) > 0:
        group_vmin = np.percentile(all_samples, pct_low)
        group_vmax = np.percentile(all_samples, pct_high)
        print(f"  Auto-scaled range (sampled): vmin={group_vmin:.4f}, vmax={group_vmax:.4f}")
    elif len(all_samples) == 0:
        print(f"  Skipping group {look}/{pol}: no valid data")
        continue

    del all_samples

    cmap = plt.colormaps[cmap_name]
    norm = mcolors.Normalize(vmin=group_vmin, vmax=group_vmax)

    try:
        font = ImageFont.truetype("DejaVuSans-Bold.ttf", 28)
    except OSError:
        font = ImageFont.load_default()

    out_gif = out_dir / f"timeseries_{look}_{pol}.gif"

    # ---- PASS 2: align + render + write, one frame at a time ----
    with imageio.get_writer(out_gif, mode='I', duration=frame_duration, loop=0) as writer:
        for f, date in items:
            arr, transform, crs = read_band_as_real(f)

            if arr.shape != ref_shape or transform != ref_transform or crs != ref_crs:
                aligned = np.zeros(ref_shape, dtype='float32')
                reproject(
                    source=arr,
                    destination=aligned,
                    src_transform=transform,
                    src_crs=crs,
                    dst_transform=ref_transform,
                    dst_crs=ref_crs,
                    resampling=Resampling.bilinear
                )
                arr = aligned
                del aligned

            arr[arr < 0] = np.nan
            nan_mask = np.isnan(arr)

            rgba = cmap(norm(np.nan_to_num(arr, nan=group_vmin)))
            rgb = (rgba[:, :, :3] * 255).astype(np.uint8)
            rgb[nan_mask] = [0, 0, 0]
            del arr, rgba

            img = Image.fromarray(rgb, mode="RGB")
            del rgb

            draw = ImageDraw.Draw(img)
            label = format_label(date)
            bbox = draw.textbbox((0, 0), label, font=font)
            text_w, text_h = bbox[2] - bbox[0], bbox[3] - bbox[1]
            margin = 10
            x = img.width - text_w - margin
            y = img.height - text_h - margin
            draw.rectangle([x - 6, y - 4, x + text_w + 6, y + text_h + 8], fill=(0, 0, 0))
            draw.text((x, y), label, fill=(255, 255, 255), font=font)

            writer.append_data(np.array(img))
            del img

    print(f"  Saved: {out_gif.name}")

print("\nDone.")

Found 4 (look, pol) group(s): [('D', 'HH'), ('D', 'HV'), ('A', 'HH'), ('A', 'HV')]

=== Group: look=D, pol=HH — 15 files ===


/tmp/ipykernel_377586/556187045.py:38: DeprecationWarning: Setting the shape on a NumPy array has been deprecated in NumPy 2.5.
As an alternative, you can create a new view using np.reshape (with copy=False if needed).
  arr = src.read(1)


  Auto-scaled range (sampled): vmin=0.0441, vmax=0.9485
  Saved: timeseries_D_HH.gif

=== Group: look=D, pol=HV — 15 files ===
  Auto-scaled range (sampled): vmin=0.0183, vmax=0.4722
  Saved: timeseries_D_HV.gif

=== Group: look=A, pol=HH — 16 files ===
  Auto-scaled range (sampled): vmin=0.0544, vmax=1.1135
  Saved: timeseries_A_HH.gif

=== Group: look=A, pol=HV — 16 files ===
  Auto-scaled range (sampled): vmin=0.0192, vmax=0.5517
  Saved: timeseries_A_HV.gif

Done.


In [ ]:
import imageio.v2 as imageio
from pathlib import Path

png_dir = Path("/home/matt/NISAR_cal_eval/Kilauea/base_images")
output_gif = png_dir / 'gif.gif'
# Sort so frames are in the right order (e.g. by filename/date)
png_files = sorted(png_dir.glob("*.png"))

for f in png_files:
    img = imageio.imread(str(f))
    print(f.name, img.shape)

frames = [imageio.imread(str(f)) for f in png_files]

imageio.mimsave(output_gif, frames, duration=0.5)  # duration = seconds per frame

NISAR_L2_PR_GSLC_006_072_D_079_4005_DHDH_A_20251127T045815_20251127T045835_X05009_N_P_J_001_HH_freqA.png (4534, 4724, 4)
NISAR_L2_PR_GSLC_006_072_D_079_4005_DHDH_A_20251127T045815_20251127T045835_X05009_N_P_J_001_HV_freqA.png (4534, 4724, 4)
NISAR_L2_PR_GSLC_006_151_A_011_4005_DHDH_A_20251202T155042_20251202T155059_X05009_N_P_J_001_HH_freqA.png (4489, 4681, 4)
NISAR_L2_PR_GSLC_006_151_A_011_4005_DHDH_A_20251202T155042_20251202T155059_X05009_N_P_J_001_HV_freqA.png (4489, 4681, 4)
NISAR_L2_PR_GSLC_008_072_D_079_4005_DHDH_A_20251221T045816_20251221T045836_X05009_N_P_J_001_HH_freqA.png (4534, 4724, 4)
NISAR_L2_PR_GSLC_008_072_D_079_4005_DHDH_A_20251221T045816_20251221T045836_X05009_N_P_J_001_HV_freqA.png (4534, 4724, 4)
NISAR_L2_PR_GSLC_008_151_A_011_4005_DHDH_A_20251226T155043_20251226T155100_X05009_N_P_J_001_HH_freqA.png (4489, 4681, 4)
NISAR_L2_PR_GSLC_008_151_A_011_4005_DHDH_A_20251226T155043_20251226T155100_X05009_N_P_J_001_HV_freqA.png (4489, 4681, 4)
NISAR_L2_PR_GSLC_009_072_D_079_4

: 

In [2]:
import re
import imageio.v2 as imageio
from pathlib import Path
from collections import defaultdict
from datetime import datetime

pattern = re.compile(
    r"NISAR_L2_PR_GSLC_"
    r"(?P<orbit>\d+)_"
    r"(?P<frame>\d+)_"
    r"(?P<direction>[AD])_"          # A = ascending, D = descending
    r".*?_"
    r"(?P<start_time>\d{8}T\d{6})_"
    r"\d{8}T\d{6}_"
    r".*?_"
    r"(?P<polarization>HH|HV)_"
    r"freq[A-Z]\.png$"
)

png_dir = Path("/home/matt/NISAR_cal_eval/Kilauea/base_images")
output_dir = png_dir  # save GIFs alongside the source PNGs
output_dir.mkdir(parents=True, exist_ok=True)


def parse_nisar_filename(filename):
    m = pattern.match(filename)
    if not m:
        return None
    d = m.groupdict()
    d["datetime"] = datetime.strptime(d["start_time"], "%Y%m%dT%H%M%S")
    return d


def group_png_files(png_dir):
    """Groups files by (direction, polarization), sorted by acquisition time."""
    groups = defaultdict(list)
    for f in Path(png_dir).glob("*.png"):
        parsed = parse_nisar_filename(f.name)
        if parsed is None:
            print(f"⚠️ Skipped unmatched file: {f.name}")
            continue
        key = (parsed["direction"], parsed["polarization"])
        groups[key].append((parsed["datetime"], f))

    for key in groups:
        groups[key].sort(key=lambda x: x[0])

    return groups


groups = group_png_files(png_dir)

for (direction, pol), files in groups.items():
    dir_label = "asc" if direction == "A" else "desc"
    output_path = output_dir / f"{dir_label}_{pol}.gif"

    with imageio.get_writer(str(output_path), mode='I', duration=0.5) as writer:
        for dt, f in files:
            img = imageio.imread(str(f))
            if img.shape[-1] == 4:
                img = img[:, :, :3]  # drop alpha channel to save memory
            writer.append_data(img)

    print(f"Saved {output_path}")

Saved /home/matt/NISAR_cal_eval/Kilauea/base_images/asc_HH.gif
Saved /home/matt/NISAR_cal_eval/Kilauea/base_images/asc_HV.gif
Saved /home/matt/NISAR_cal_eval/Kilauea/base_images/desc_HV.gif
Saved /home/matt/NISAR_cal_eval/Kilauea/base_images/desc_HH.gif
